In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

#storage_details
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<YOUR_KEY>"

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/customers"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dimensions/dim_customers"


In [0]:
customer_df = spark.read.format("delta").load(silver_path)

In [0]:
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
required_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]

missing_columns = [c for c in required_columns
                   if c not in customer_df.columns]

if missing_columns:
    raise Exception(f"Missing columns: {missing_columns}")

In [0]:
dim_columns = (
    customer_df.select(required_columns)
    .dropDuplicates(["customer_id"])
      .withColumn("customer_key", F.monotonically_increasing_id())
)

In [0]:
display(dim_columns, limit = 10)

customer_id,customer_unique_id,customer_city,customer_state,customer_key
00050bf6e01e69d5c0fd612f1bcfb69c,e3cf594a99e810f58af53ed4820f25e5,ijui,rs,0
000598caf2ef4117407665ac33275130,7e0516b486e92ed3f3afdd6d1276cfbd,oliveira,mg,1
000bf8121c3412d3057d32371c5d3395,1bc9b2dad6aefbfbc011508e34c8adfc,jacarei,sp,2
00114026c1b7b52ab1773f317ef4880b,f4dc0a81a11d3d270ccf5a9c4b5b187b,rio de janeiro,rj,3
0013cd8e350a7cc76873441e431dd5ee,334fed5abcee3aa96c13f1432703e1fd,sao paulo,sp,4
0015bc9fd2d5395446143e8b215d7c75,490c854539b21598cfbbac518ca25788,sao jose dos campos,sp,5
0015f7887e2fde13ddaa7b8e385af919,866c923cde750dfc8cfbcf9d5ced0ee4,mage,rj,6
001a57041f56400917a187dd74e6cbc1,163b27a06a32c2fa565927170b59b5d4,sao paulo,sp,7
001df1ee5c36767aa607001ab1a13a06,46b44ab325f78e5bb3dc0bbef1082082,sao paulo,sp,8
001f150aebb5d897f2059b0460c38449,0f88eb431888ffb9d726252b7ac8cefe,campo grande,ms,9


In [0]:
dim_columns.write.format("delta").save(gold_path)